# Domux safety-gate experiment

This notebook runs a real pinned Hugging Face snapshot of `iFlytekOpenSource/Domux` on 48 public synthetic smart-home safety cases. It produces a small public evidence bundle (raw model outputs, metadata, report, and log) but never stores or prints a Hugging Face token. Use a GPU runtime.

In [ ]:
!nvidia-smi
!python --version
!pip -q install 'transformers>=5.0.0' 'accelerate>=1.10.0' 'bitsandbytes>=0.49.0' 'huggingface_hub>=1.0.0'

## Hugging Face authentication

Accept the Gemma terms on the Domux model page first. The official device-login widget opens a Hugging Face authorization flow. Review its permissions before authorizing. Do not paste a token into any code cell, output, screenshot, Discussion, or GitHub file.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!git clone --depth 1 --branch case/domux-safety-gate https://github.com/yangmengze608-afk/domux.git /content/domux
%cd /content/domux/cases/domux-safety-gate
!python -m unittest -v test_safety_gate.py test_dataset.py test_evaluate_safety.py

## Smoke test

Run two samples first. If the free GPU cannot load the NF4 model, stop here; do not switch to a paid runtime without explicit approval.

In [ ]:
!python run_transformers.py --dataset example_safety_commands.jsonl --output results/smoke.jsonl --quantization nf4 --limit 2 --warmup 1 --seed 20260825
!sed -n '1,2p' results/smoke.jsonl
!cat results/smoke.metadata.json

## Full 48-case run, evaluation, and public evidence

The final cell copies only raw outputs, run metadata, and the recomputable report into `evidence/`. It never copies model weights, caches, or credentials. Download the resulting zip and add these evidence files to the repository before publishing.

In [ ]:
!python run_transformers.py --dataset example_safety_commands.jsonl --output results/domux_raw.jsonl --quantization nf4 --warmup 2 --seed 20260825
!python evaluate_safety.py --dataset example_safety_commands.jsonl --responses results/domux_raw.jsonl --output results/safety_report.json

In [ ]:
import json
from pathlib import Path
report = json.loads(Path('results/safety_report.json').read_text())
{key: value for key, value in report.items() if key != 'details'}

In [ ]:
from pathlib import Path
import shutil

evidence = Path('evidence')
evidence.mkdir(exist_ok=True)
for name in ('domux_raw.jsonl', 'domux_raw.metadata.json', 'safety_report.json'):
    shutil.copy2(Path('results') / name, evidence / name)
shutil.make_archive('/content/domux-safety-gate-evidence', 'zip', evidence)
print('/content/domux-safety-gate-evidence.zip contains raw outputs, metadata, and report only; no model weights or token.')